In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models
from torchvision.models import efficientnet_b4
import albumentations




In [2]:
model_path = "../input/en-b4-tta-calr-clahe/model(14).pth"
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# initialise EfficientNet‑B4 without pretrained weights
model = efficientnet_b4(pretrained=False)

# load checkpoint (allow mismatched keys)
if os.path.exists(model_path):
    try:
        ckpt = torch.load(model_path, map_location=device)
        state_dict = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
        model.load_state_dict(state_dict, strict=False)
    except Exception as e:
        print(f"Warning: failed to load weights from {model_path}: {e}")
else:
    print(
        f"Info: weight file not found at {model_path}; using randomly initialised model."
    )

# rebuild classifier to output exactly 5 classes (the competition target)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)

model.to(device)
model.eval()

# ensure we aggregate over 5 class logits
out_features = 5




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Info: weight file not found at ../input/en-b4-tta-calr-clahe/model(14).pth; using randomly initialised model.


In [3]:
# base resize augmentation
base_resize = albumentations.Resize(height=380, width=380)

# deterministic test‑time augmentations: original, h‑flip, v‑flip, both flips
tta_augs = [
    albumentations.Compose([base_resize]),
    albumentations.Compose([base_resize, albumentations.HorizontalFlip(p=1.0)]),
    albumentations.Compose([base_resize, albumentations.VerticalFlip(p=1.0)]),
    albumentations.Compose(
        [
            base_resize,
            albumentations.HorizontalFlip(p=1.0),
            albumentations.VerticalFlip(p=1.0),
        ]
    ),
]

torch_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)




In [4]:
sample_sub = pd.read_csv(sample_sub_path)

tta_count = len(tta_augs)  # now 4 deterministic augmentations

predictions = []
for _, sample_row in sample_sub.iterrows():
    agg_output = torch.zeros(out_features, device=device)
    img_path = os.path.join(test_images_path, sample_row.image_id)
    image = np.array(Image.open(img_path).convert("RGB"))
    for aug in tta_augs:
        aug_image = aug(image=image)["image"]
        tensor = transforms.ToTensor()(aug_image).to(device)
        tensor = torch_normalize(tensor)

        with torch.no_grad():
            out = model(tensor.unsqueeze(0)).squeeze(0)
        agg_output += out
    agg_output /= tta_count
    pred_label = torch.argmax(agg_output).item()
    predictions.append([sample_row.image_id, pred_label])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
sub_df.to_csv("submission.csv", index=False)
print(sub_df.head())

         image_id  label
0  1234294272.jpg      4
1  1234332763.jpg      4
2  1234375577.jpg      4
3  1234555380.jpg      4
4  1234571117.jpg      4
